In [3]:
import pandas as pd
import re
import numpy as np

In [4]:
df = pd.read_csv("questionnaire_factor_associations_combined.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'questionnaire_factor_associations_combined.csv'

In [ ]:
def remove_k_all(x):

    x = str(x)

    # remove _K1, _K2, ...
    x = re.sub(r"_K\d+$", "", x)

    # remove trailing _K (important!)
    x = re.sub(r"_K$", "", x)

    return x

df["score_clean"] = df["score_base"].apply(remove_k_all)
df["factor_clean"] = df["factor_base"].apply(remove_k_all)

In [ ]:
def weighted(group):

    assoc = group["association"].values
    weights = group["total_n"].values

    return pd.Series({
        "association": np.average(assoc, weights=weights),
        "total_n": weights.sum(),
        "num_points": len(group),
        "factor_type": group["factor_type"].iloc[0]
    })

In [ ]:
merged = (
    df
    .groupby([
        "questionnaire",
        "score_clean",
        "factor_clean"
    ])
    .apply(weighted)
    .reset_index()
)

In [ ]:
merged = merged.rename(columns={
    "score_clean": "score",
    "factor_clean": "factor"
})

In [ ]:
merged["abs_assoc"] = merged["association"].abs()

merged = merged.sort_values(
    ["questionnaire", "abs_assoc"],
    ascending=[True, False]
)

In [ ]:
for q in merged["questionnaire"].unique():

    print("\n" + "=" * 80)
    print(q)
    print("=" * 80)

    print(
        merged[merged["questionnaire"] == q]
        .head(20)[
            ["score", "factor", "association", "total_n"]
        ]
    )


In [ ]:
merged.to_csv(
    "questionnaire_factor_associations_FINAL_MERGED.csv",
    index=False
)

print("\nSaved FINAL merged dataset.")